# House Prices: Advanced Regression Techniques
### Home Price Prediction (Kaggle Competition)
**Author:** Eduard Trubichkin  
**Date:** march 2026  
**Description:** Complete data analysis, preprocessing, hyperparameter tuning, and linear models (Ridge, Lasso, ElasticNet).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge, LassoCV, Lasso, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_log_error

We create 2 objects from the test and training data.

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

Let's do a general analysis of the `train` sample. Let's determine if there are gaps in the data.

In [ ]:
train.info()

Let's take a closer look at the price values and determine if there are any outliers.

In [ ]:
train['SalePrice'].describe()

There are a lot of omissions in the training sample, and there is also an outlier (a large gap between 75% and max). Let's start working with the passes. Let's determine how many columns have gaps and the number of gaps in each column.

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

Let's determine the number of passes as a percentage.

In [ ]:
missing_percent = train.isnull().sum() / len(train) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
print(missing_percent)

There should be no gaps in the training sample. When filling in the gaps, you need to rely on the file `data_description.txt`, since some omissions may mean the absence of some feature, which is also an important parameter. For example, the `PoolQC` parameter has the largest number of gaps, but if we look at `data_description.txt` `PoolQC` means the quality of the pool, and skipping `NA` means there is no pool in the house.  
But since pandas works in such a way that values like `NA` are converted to omissions, it is necessary to manually correct omissions to `NoPool`, for example.   
In some parameters, it is necessary to fill in the gaps with the median, as, for example, in the case of `LotFrontage`. Just group it by some other parameter.   
In cases where the parameter is omitted in only one or two houses, you can completely delete the row, this will not affect anything.   

In [ ]:
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
train['MiscFeature'] = train['MiscFeature'].fillna('NoMiscFeature')
train['Alley'] = train['Alley'].fillna('NoAlley')
train['Fence'] = train['Fence'].fillna('NoFence')
train['MasVnrType'] = train['MasVnrType'].fillna('NoMasVnrType')
train['FireplaceQu'] = train['FireplaceQu'].fillna('NoFireplace')

Next is the `LotFrontage` parameter. As mentioned earlier, we will use the median by district to fill in the gaps. If there are no `LotFrontage` values in one or more districts and, accordingly, it will not be possible to calculate the median, then we will fill in such gaps with the total median.

In [ ]:
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

if train['LotFrontage'].isnull().any():
    global_median = train['LotFrontage'].median()
    train['LotFrontage'] = train['LotFrontage'].fillna(global_median)

The following is a list of garage parameters. The `GarageYrBlt` parameter is numeric, and omissions can be replaced with zeros. The other parameters are string parameters. Let's replace the omissions with `NoGarage`.

In [ ]:
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)

garage_str_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_str_cols:
    train[col] = train[col].fillna('NoGarage')

Next are the basement parameters. Working with these parameters is similar to working with garage parameters.

In [ ]:
bsmt_str_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
for col in bsmt_str_cols:
    train[col] = train[col].fillna('NoBasement')

There are 2 columns left with gaps: `MasVnrArea` and `Electrical`. In the first case, we will replace all the omissions with zeros, in the second case, we will simply delete the only line where there is a skip.

In [ ]:
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train.dropna(subset='Electrical', inplace=True)

Let's check if all the gaps are filled now.

In [ ]:
print(train.isnull().sum().sum())

Now let's work with the outlier. As mentioned earlier, there is a large gap between 75% and max in the training sample. This means that there are several very expensive houses in the sample. And such a gap can confuse the model, so we'll add another column containing logarithmic prices. It is on the logarithmic price that the model will be trained.

In [ ]:
train['SalePrice_log'] = np.log1p(train['SalePrice'])

Now let's move on to encoding the parameters. All the parameters that are in `train.csv` must be represented by numbers, since only numbers understand the model. For this, the `preprocessor` is used, specifically its `fit_transform()` method, which trains the preprocessor and converts all data to a numeric type.   

Before creating a preprocessor, it is necessary to implement encoders, which are needed to convert strings into numbers that the model understands. In addition to encoders, you will also need to implement lists of column names that define the data type.   

There will be 3 such columns in total: numeric, ordinal, and nominal:  
- numeric signs - signs whose values are numbers, you can not touch them and leave them as they are  
- ordinal signs - signs whose values can be estimated, distributed from the worst to the best; for example, the exterior coating of a house or the quality of kitchen decoration 
- nominal - signs whose values cannot be evaluated relative to each other; for example, the color of a fence, it cannot be said that a green fence is better than a red one   

Ordinal features include the following parameters:

In [ ]:
ordinal_mapping = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NoBasement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NoGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NoPool', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Fence': ['NoFence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

In the dictionary, they place all ordinal signs along with values arranged in order from worst to best.  
The dictionary is necessary in order to create 2 arrays for ordinary features based on it: the first array is a list of columns, the second array is an array of arrays with the correct location of values. Thus, the dictionary will combine these 2 arrays, and if it is necessary to make changes to some array, this can be done in the dictionary and the changes will "spread" to the 2 arrays.  

Initialization of arrays with numeric, ordinal, and nominal attributes:

In [ ]:
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns.to_list()
numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]

ordinal_features = list(ordinal_mapping.keys())

categorical_features = train.select_dtypes(include=['str']).columns.to_list()
categorical_features = [col for col in categorical_features if col not in ordinal_features]

We will also create an array that includes all arrays with ordinal values:

In [ ]:
categories_list = [ordinal_mapping[col] for col in ordinal_features]

As mentioned earlier, these arrays are needed for encoders, and encoders are needed for preprocessor. Therefore, we proceed to the implementation of encoders. Let's create an OrdinalEncoder for ordinal features and an OneHotEncoder for nominal features. Encoders are needed to convert strings to numbers. For example, OrdinalEncoder, the name of an array with all values in the correct order, equates the values to the numbers corresponding to the index. That's why it's important to put them in the right order, from worst to best. And OneHotEncoder creates binary columns for nominal signs. There are as many columns as there are values for the attribute. If, for example, the color of the fence can have the values: "green", "yellow" and "red". And some house will have a green fence, then OneHotEncoder will display it as `[1, 0, 0]`. These encoders are then used to create and train the preprocessor.   

An array of arrays with the correct location of values is passed to OdrinalEncoder. There are also rules that establish behavior when encountering values that have not been encountered before. In this case, when encountering an unfamiliar value, the number "-1" is equated to it.  
OneHotEncoder sets the behavior for nominal attributes. If a value is found that was not in the training dataset, it is simply ignored.

In [ ]:
ordinal_encoder = OrdinalEncoder(
    categories=categories_list,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

Now you can implement a preprocessor. This is done using ColumnTransformer. All encoders and all 3 arrays must be passed to it. In this way, the preprocessor will become a tool that transforms raw data into data on which the model can be trained and tested.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('cat', categorical_encoder, categorical_features)
    ]
)

Next, to convert the `train`, you need to use the `fit_transform()` method, where you need to pass this very `train`. But before that, we will remove unnecessary columns from it: 'Id', 'SalePrice' and 'SalePrice_log', as well as logarithmic prices in a separate variable.

In [ ]:
X = train.drop(['Id', 'SalePrice', 'SalePrice_log'], axis=1)
Y = train['SalePrice_log']

X_processed = preprocessor.fit_transform(X)

Now we have a full-fledged dataset on which to train the model. Let's divide this dataset into parts, one part will be for training, the second part for validation to make sure that the model is working correctly. This is done using the `train_test_split()` function:

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_processed, Y, test_size=0.2, random_state=42
)

Now everything is ready to create and train the model.

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X_train, Y_train)
preds_log = model.predict(X_val)

Since we taught the model to predict the natural logarithm of the price, all values need to be converted back to absolute values.    
And also use RMSLE to check the quality of the model.

In [ ]:
Y_orig = np.expm1(Y_val)
preds_orig = np.expm1(preds_log)

rmsle = np.sqrt(mean_squared_log_error(Y_orig, preds_orig))
print(rmsle)

Let's try to improve the model. First, let's try using different alpha values. We will use **cross-validation** for this. Cross-validation is a method of evaluating a model in which all training data is divided into "K" equal parts (folds). One fold remains to be checked, and training is conducted on the other folds. And so on until each of the folds is in the role of validation. For example, if we have a training dataset of 100 lines, and K=5, then the dataset is divided into 5 folds (20 lines each). The first fold (the first 20 lines) remains for verification, 2, 3, 4, 5 folds (80 lines) are a training sample. In the next iteration, the second fold will be a validation fold, and the model will be trained on the 1st, 3rd, 4th, and 5th folds. And so on until all the folds have been validated.    
After each iteration, the RMSLE must be calculated. After all iterations, the average RMSLE is calculated (we add all 5 values and divide by the number). The arithmetic mean will be the result of cross-validation.

Create an array of eight alpha values. Next, let's start going through them in a loop. At each iteration:  
- Creating a ridge model with an alpha value that corresponds to the iteration index
- Create the `scores` variable, which contains the result of the `cross_val_score()` function. This function evaluates the model using the cross-validation method, the principle of which is described above. The model created earlier in the same iteration is passed to it, `X_processed` and `Y` (training dataset), `cv` (argument indicating the number of folds/splits) and `scoring=neg_mean_squared_error` (what the scores variable will store). The principle of operation is as follows: the `cross_val_score()` function splits the training data into 5 parts. Then he trains the model on it. `cv=5` means that there are 5 folds in total, and correspondingly 5 iterations of learning in the function per iteration of the entire cycle. The result of each training iteration is calculated in a negative MSLE value. Thus, at each iteration of the loop, the `scores` variable contains an array of 5 negative MSLE values (which are the result of evaluating the training of the model using the cross-validation method). That is, we create a model separately, and training and predictions are already happening directly inside the function `cross_val_score()`. That is, it replaces `model.fit()` and `model.predict()`
- Conversion of the array to a full-fledged RMSLE. Calculate the average and get rid of the minus
- Add the arithmetic mean to the array, which will contain all the arithmetic averages for each alpha value 
- Display the results of cross-validation for all alpha values


You should also pay attention to the fact that the `scoring=neg_mean_squared_error` argument was passed to the `cross_val_score()` function, since the passed `Y` is the logarithmic prices. And the `neg_mean_squared_error` is what is calculated at each training iteration, that is, a negative MSE value. And since we transmit logarithmic prices, the metric should also be calculated in MSE. If we passed `neg_mean_squared_log_error` to the function, the function would calculate negative MSLE values. That is, it would logarithm the logarithmic data already. This is an important nuance.

In [ ]:
alphas = [0.1, 0.5, 1, 5, 10, 20, 50, 100]
cv_scores = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_processed, Y, cv=5, scoring='neg_mean_squared_error')
    rmsle = np.sqrt(-scores.mean())
    cv_scores.append(rmsle)
    print(f"alpha={alpha:5.1f}, RMSLE={rmsle:.4f}")

Let's calculate the best alpha value of all:

In [ ]:
best_alpha = alphas[np.argmin(cv_scores)]
print(best_alpha)

Now let's create a submission.csv using the Ridge-model with the best alpha value. But before that, let's fill in the gaps in the `test`.

In [ ]:
test['Alley'] = test['Alley'].fillna('NoAlley')
test['PoolQC'] = test['PoolQC'].fillna('NoPool')
test['MiscFeature'] = test['MiscFeature'].fillna('NoMiscFeature')
test['Fence'] = test['Fence'].fillna('NoFence')
test['MasVnrType'] = test['MasVnrType'].fillna('NoMasVnrType')
test['FireplaceQu'] = test['FireplaceQu'].fillna('NoFireplace')

garage_str = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_str:
    test[col] = test[col].fillna('NoGarage')

test['GarageYrBlt'] = test['GarageYrBlt'].fillna(0)
test['GarageArea'] = test['GarageArea'].fillna(0)
test['GarageCars'] = test['GarageCars'].fillna(0)

bsmt_str = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in bsmt_str:
    test[col] = test[col].fillna('NoBasement')

num_bsmt = ['BsmtFullBath', 'BsmtHalfBath', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF1', 'BsmtFinSF2']
for col in num_bsmt:
    test[col] = test[col].fillna(0)

lot_medians = train.groupby('Neighborhood')['LotFrontage'].median()
test['LotFrontage'] = test.apply(
    lambda row: lot_medians[row['Neighborhood']] if pd.isnull(row['LotFrontage']) else row['LotFrontage'],
    axis=1
)
global_median = train['LotFrontage'].median()
test['LotFrontage'] = test['LotFrontage'].fillna(global_median)

test['MasVnrArea'] = test['MasVnrArea'].fillna(0)

cat_cols = ['MSZoning', 'Functional', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'SaleType']
for col in cat_cols:
    if test[col].isnull().any():
        mode_val = train[col].mode()[0]
        test[col] = test[col].fillna(mode_val)

print(test.isnull().sum().sum())

Let's reduce the `test` to a numerical matrix using a trained preprocessor and train the model.

In [ ]:
model_v2 = Ridge(alpha=best_alpha)
model_v2.fit(X_processed, Y)

test_ids = test['Id']

X_test = test.drop(['Id'], axis=1)
X_test_processed = preprocessor.transform(X_test)

preds_log = model_v2.predict(X_test_processed)
preds_log_orig = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds_log_orig
})

submission.to_csv('submission_2.csv', index=False)

When using the Ridge model again, but with alpha=10, the result on Kaggle has changed. `Score=0.13290`.  

Changing the alpha value gave a small increase in quality, but not strong enough, so let's try `feature engineering`.   

Add additional columns for `test` and `train'. You can add: total area, age of the house at the time of sale, age of renovation, garage, basement, fireplace, total number of bathrooms. All these signs are numeric, but it is important to pay attention to the fact that of these 7 numeric signs, 3 are binary. However, they are numeric, not nominal, because if you make these signs nominal, you will get 2 columns each. And these columns will be mirrored (if one has 1, then the other must have 0). This creates redundancy (multicollinearity), which is harmful for many models. Therefore, all the listed signs will be numeric.

In [ ]:
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

train['HasGarage'] = (train['GarageArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)

train['HasBasement'] = (train['TotalBsmtSF'] > 0).astype(int)
test['HasBasement'] = (test['TotalBsmtSF'] > 0).astype(int)

train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)

train['TotalBath'] = train['FullBath'] + 0.5*train['HalfBath'] + train['BsmtFullBath'] + 0.5*train['BsmtHalfBath']
test['TotalBath'] = test['FullBath'] + 0.5*test['HalfBath'] + test['BsmtFullBath'] + 0.5*test['BsmtHalfBath']

We have added new attributes for `train` and `test`, they are numeric. Now you need to edit the `numeric_features` by adding new features there. Encoders do not need to be recreated, as the changes do not relate to ordinal and nominal attributes.

In [ ]:
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns.to_list()
numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]

Let's recreate and retrain the preprocessor.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('cat', categorical_encoder, categorical_features)
    ]
)

X = train.drop(['Id', 'SalePrice', 'SalePrice_log'], axis=1)
Y = train['SalePrice_log']

X_processed = preprocessor.fit_transform(X)

Since the data has changed, it is worth analyzing again which alpha value will be the best.

In [ ]:
alphas = [0.1, 0.5, 1, 5, 10, 20, 50, 100]
cv_scores = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_processed, Y, cv=5, scoring='neg_mean_squared_error')
    rmsle = np.sqrt(-scores.mean())
    cv_scores.append(rmsle)
    print(f"alpha={alpha:5.1f}, RMSLE={rmsle:.4f}")

best_alpha = alphas[np.argmin(cv_scores)]

Now you can start training the Ridge model.

In [ ]:
model_v3 = Ridge(alpha=best_alpha)
model_v3.fit(X_processed, Y)

Now let's apply the trained model to the `test`.

In [ ]:
X_test = test.drop(['Id'], axis=1)
X_test_processed = preprocessor.transform(X_test)

preds_log = model_v3.predict(X_test_processed)
preds_orig = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds_orig
})

submission.to_csv('submission_3.csv', index=False)

The result of the current model has become worse compared to the previous one. On Kaggle this model has a `score = 0.13304`.   

Now let's try the Lasso model, a linear regression model with L1 regularization. Its special feature is that, unlike Ridge, it can reset uninformative data. Therefore, it will operate on data from feature engineering.   
The principle is similar to previous attempts. We already have data, encoders, preprocessor, and accordingly we already have a worker `X_processed` and `Y`.    

Let's look at the alpha selection process for Lasso. This is done using `LassoCV`. There is also a `RidgeCV`, but last time we manually selected alpha to see how it is done. Now let's use a more convenient automatic tool.

A number of arguments are passed to LassoCV:  
- alphas - range of values. In this case, the `logspace` function from NumPy is used, which takes 3 numeric arguments: the value of the left boundary of the range (power of 10), the value of the right boundary of the range (power of 10) and the number of values in the range. In this case, we use the entry `np.logspace(-1, 4, 50)`, which means 50 values from 10⁻⁴ before 10⁰
- cv - number of folds for cross validation 
- random_state - fixes a random number generator for reproducibility, divided into folds
- max_iter - the maximum number of iterations of the algorithm 
- n_jobs - the number of parallel threads to speed up calculations (-1 means to use all cores)

In [ ]:
lasso_cv = LassoCV(
    alphas = np.logspace(-4, 0, 50),
    cv=5,
    random_state=42,
    max_iter=10000,
    n_jobs=-1
)

Now you need to start the calculation process. This is done using the `fit()` method. After `lasso_cv` is trained, the best alpha is stored in the attribute `lasso_cv.alpha_`. You can output the best alpha. And the `lasso_cv.coef` attribute stores a vector of coefficients/weights for each feature.

In [ ]:
lasso_cv.fit(X_processed, Y)
print(f"The best alpha for Lasso: {lasso_cv.alpha_:.6f}")
print(f"Number of non-zero weights: {np.sum(lasso_cv.coef_ != 0)} from {len(lasso_cv.coef_)}")

The best alpha for Lasso: 0.000543   
Number of non-zero weights: 98 из 233    
We see that Lasso zeroes out too many features. Most likely, the result will be depressing, but it's worth a try.

Now that we have X_processed, Y and the best alpha, we can start training the model.

In [ ]:
model_v4_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
model_v4_lasso.fit(X_processed, Y)

`X_test_processed` (the test dataset after the preprocessor) is already there, so we can start testing the model and uploading the results.

In [ ]:
preds_log = model_v4_lasso.predict(X_test_processed)
preds_orig = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds_orig
})

submission.to_csv('submission_4.csv', index=False)

The result was excellent. This is the best result of all the attempts. `Score = 0.13239`.   

Let's try the ElasticNet model. The preprocessor is already ready in the previous steps, it remains to select the best alpha and train the model. Unlike `LassoCV`, another argument is added to `ElasticNetCV` - `l1_ratio`. This is the proportion of L1 regularization in ElasticNet. If `l1_ratio = 1`, it means that ElasticNet is in fact a Lasso (linear model with L1 regularization), if `l1_ratio = 0`, it means that ElasticNet is in fact a Ridge (linear model with L2 regularization). In ElasticNetCV, the `l1_ratio` coefficient is selected through cross-validation, just like alpha.

In [ ]:
elastic_net_cv = ElasticNetCV(
    alphas = np.logspace(-4, 0, 50),
    l1_ratio = np.linspace(0, 1, 100),
    cv=10,
    random_state=42,
    max_iter=10000,
    n_jobs=-1
)

Next, we'll train `elastic_net_cv` and see which parameters were selected:

In [ ]:
elastic_net_cv.fit(X_processed, Y)

print(f"Best alpha: {elastic_net_cv.alpha_:.6f}")
print(f"Best l1_ratio: {elastic_net_cv.l1_ratio_:.6f}")
print(f"Number of non-zero weights: {np.sum(elastic_net_cv.coef_ != 0)} from {len(elastic_net_cv.coef_)}")

Learning outcomes:   
1) Best alpha: 0.000791   
2) Best l1_ratio: 0.505051    
3) Number of non-zero weights: 113 from 233    

Now you can proceed to creating and training an ElasticNet model.  

In [ ]:
model_v5_elastic = ElasticNet(alpha=elastic_net_cv.alpha_, l1_ratio=elastic_net_cv.l1_ratio_, max_iter=10000)
model_v5_elastic.fit(X_processed, Y)

Moving on to the predictions on the test dataset:

In [ ]:
preds_log = model_v5_elastic.predict(X_test_processed)
preds_orig = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds_orig
})

submission.to_csv('submission_5.csv', index=False)

The indicators have become worse. ElasticNet did not give an improvement, `score = 0.13270`.